In [4]:
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.types import Command,interrupt
from langchain_core.messages import AIMessage,HumanMessage,ToolMessage
from typing import Annotated,TypedDict
import os
import sqlite3
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

True

In [14]:
class ApprovalState(TypedDict):
    task:str
    plan:str
    result:str
    approved:bool

def plan_task(state:ApprovalState)->dict:
    plan=f"I will execute the task {state['task']} using the following steps: 1.setup 2.execute 3.verify"
    print(f"Generated Plan: {plan}")

    approval=interrupt(
    {
        "question":"Do you approve this plan ?",
        "plan": plan,
        "options":["yes","no","verify"]
    })
    return {"plan":plan, "approved":approval=="yes"}

def execute_task(state:ApprovalState)->dict:
    if not state['approved']:
        return {"msg":"Cancelled by the USER"}
    result=f"Successfully executed {state['task']}"

    return {"result":result}

db_path="Memory/HTL.db"
conn=sqlite3.connect(db_path,check_same_thread=False)
memory=SqliteSaver(conn)

graph=StateGraph(ApprovalState)

graph.add_node("plan",plan_task)
graph.add_node("execute",execute_task)
graph.add_edge(START,"plan")
graph.add_edge("plan","execute")
graph.add_edge("execute",END)

app=graph.compile(checkpointer=memory)


In [15]:
config={"configurable":{"thread_id":"my-user123"}}
result=app.invoke({
    "task":"Migrate Database","plan":"","result":"","approved":"False"
},config)

print(f"Result is {result}")
state=app.get_state(config)
print(f"State is : {state.values['task']}")

final_result=app.invoke((Command(resume="yes")),config)
print(f"Final-Result:{final_result}")


Generated Plan: I will execute the task Migrate Database using the following steps: 1.setup 2.execute 3.verify
Result is {'task': 'Migrate Database', 'plan': '', 'result': '', 'approved': 'False', '__interrupt__': [Interrupt(value={'question': 'Do you approve this plan ?', 'plan': 'I will execute the task Migrate Database using the following steps: 1.setup 2.execute 3.verify', 'options': ['yes', 'no', 'verify']}, id='53550b6494bd2bd5db737481e843951d', response_schema=None)]}
State is : Migrate Database
Generated Plan: I will execute the task Migrate Database using the following steps: 1.setup 2.execute 3.verify
Final-Result:{'task': 'Migrate Database', 'plan': 'I will execute the task Migrate Database using the following steps: 1.setup 2.execute 3.verify', 'result': 'Successfully executed Migrate Database', 'approved': True}


In [10]:
from langgraph.types import Command,interrupt
from typing import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.sqlite import SqliteSaver

db_path="Memory/HTL.db"
conn=sqlite3.connect(db_path,check_same_thread=False)
memory=SqliteSaver(conn=conn)

class AgentState(TypedDict):
    task: str
    plan: str
    approval: bool
    result: str

def plan_task(state:AgentState)->dict:
    plan=(f"Plan for '{state['task']}':\n"
        "  Step 1 — Validate inputs\n"
        "  Step 2 — Execute operation\n"
        "  Step 3 — Verify results"
        )
    print(f"Generated PLAN : {plan} ")
    approval=interrupt(
        {
            "question": "Do you approve this plan",
            "plan": plan,
            "options": ['yes','no','modify'],
        }
    )

    print(f"Human Response: {approval}")
    return {"plan":plan,"approval":(approval=="yes")}

def execute_task(state:AgentState)->dict:
    if not state['approval']:
        result=f"PLAN for the task: {state['task']} not approved"
    else:
        result=f"The task execution is Successfull"
    return {"result": result}

graph=StateGraph(AgentState)
graph.add_node("plan",plan_task)
graph.add_node("execute",execute_task)
graph.add_edge(START,"plan")
graph.add_edge("plan","execute")
graph.add_edge("execute",END)

app=graph.compile(checkpointer=memory)



In [11]:
config={"configurable":{"thread_id":"user-123"}}
result=app.invoke({"task":"Deploy Infra migration","plan":"","approval":False,"result":""},config=config)

human_answer=input("Choose YES to approve,Choose NO to reject and MODIFY to change th plan: ").strip().lower()
final_result=app.invoke(Command(resume=human_answer),config)

print(final_result)

Generated PLAN : Plan for 'Deploy Infra migration':
  Step 1 — Validate inputs
  Step 2 — Execute operation
  Step 3 — Verify results 
Generated PLAN : Plan for 'Deploy Infra migration':
  Step 1 — Validate inputs
  Step 2 — Execute operation
  Step 3 — Verify results 
Human Response: no
{'task': 'Deploy Infra migration', 'plan': "Plan for 'Deploy Infra migration':\n  Step 1 — Validate inputs\n  Step 2 — Execute operation\n  Step 3 — Verify results", 'approval': False, 'result': 'PLAN for the task: Deploy Infra migration not approved'}


In [14]:
class Payment(TypedDict):
    amount: float
    merchant: str
    status: str
db_path="Memory/HTL.db"
conn=sqlite3.connect(db_path,check_same_thread=False)
memory=SqliteSaver(conn=conn)
def validate_payment(state:Payment):
    print(f"Validating payment of Amount: {state['amount']} to Merchant: {state['merchant']}")
    return {"status":"Validated Payment"}

def process_payment(state:Payment):
     print(f"Payment of Amount: {state['amount']} to Merchant: {state['merchant']} Made")
     return {"status":"Payment Done"}

graph=StateGraph(Payment)

graph.add_node("validate",validate_payment)
graph.add_node("process",process_payment)
graph.add_edge(START,"validate")
graph.add_edge("validate","process")
graph.add_edge("process",END)

app=graph.compile(checkpointer=memory,interrupt_before=["process"])
config={"configurable":{"thread_id":"user-456"}}

result=app.invoke({"amount": 5000.0, "merchant":"AWS", "status":"pending"},config=config)
print(f"State:{result['status']}")

final_result=app.invoke(None,config)
print(f"Result: {final_result['status']}")







Validating payment of Amount: 5000.0 to Merchant: AWS
State:Validated Payment
Payment of Amount: 5000.0 to Merchant: AWS Made
Result: Payment Done


In [20]:
import sqlite3
from langgraph.graph import StateGraph,START,END,MessagesState
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.messages import HumanMessage


def step1(state:MessagesState)->dict:
    return {"messages":[{"role":"assistant","content":"Step 1 completed execution"}]}

def step2(state:MessagesState)->dict:
    return {"messages":[{"role":"assistant","content":"Step 1 completed execution"}]}

graph=StateGraph(MessagesState)
graph.add_node("step1",step1)
graph.add_node("step2",step2)

graph.add_edge(START,"step1")
graph.add_edge("step1","step2")
graph.add_edge("step2",END)

app=graph.compile()

for chunks in app.stream({"messages":[HumanMessage(content="GO")]},stream_mode=['values','updates']):
    print(f"Chunk: {chunks}")

Chunk: ('values', {'messages': [HumanMessage(content='GO', additional_kwargs={}, response_metadata={}, id='1859efdf-fac5-4df6-8854-f1896fbabcc4')]})
Chunk: ('updates', {'step1': {'messages': [{'role': 'assistant', 'content': 'Step 1 completed execution'}]}})
Chunk: ('values', {'messages': [HumanMessage(content='GO', additional_kwargs={}, response_metadata={}, id='1859efdf-fac5-4df6-8854-f1896fbabcc4'), AIMessage(content='Step 1 completed execution', additional_kwargs={}, response_metadata={}, id='6ebdf086-8853-45ce-b855-044d17832694', tool_calls=[], invalid_tool_calls=[])]})
Chunk: ('updates', {'step2': {'messages': [{'role': 'assistant', 'content': 'Step 1 completed execution'}]}})
Chunk: ('values', {'messages': [HumanMessage(content='GO', additional_kwargs={}, response_metadata={}, id='1859efdf-fac5-4df6-8854-f1896fbabcc4'), AIMessage(content='Step 1 completed execution', additional_kwargs={}, response_metadata={}, id='6ebdf086-8853-45ce-b855-044d17832694', tool_calls=[], invalid_too

In [22]:
from langgraph.config import get_stream_writer
class ProcessedState(TypedDict):
    item: list
    processed: list

def process_items(state:ProcessedState)->dict:
    writer=get_stream_writer()
    processed=[]

    for i,items in enumerate(state['item']):
        result=f"Processed {items}"
        processed.append(result)

        writer({'state':'progress','item':items,'done':i,'total':len(state['item'])})

    return {"processed":result}

graph=StateGraph(ProcessedState)
graph.add_node("process",process_items)

graph.add_edge(START,"process")
graph.add_edge("process",END)

app=graph.compile()

for chunks in app.stream({"item":["a","b","c"],"processed":""},stream_mode="custom"):
    print(f"State: {chunks}")


State: {'state': 'progress', 'item': 'a', 'done': 0, 'total': 3}
State: {'state': 'progress', 'item': 'b', 'done': 1, 'total': 3}
State: {'state': 'progress', 'item': 'c', 'done': 2, 'total': 3}


In [3]:
from langchain_openai import ChatOpenAI
from langgraph.graph import MessagesState,StateGraph,START,END

llm=ChatOpenAI(model='gpt-4o-mini',temperature=0)

def node(state:MessagesState)->dict:
    result=llm.invoke(state['messages'])
    return {"messages":result}

graph=StateGraph(MessagesState)
graph.add_node("node",node)
graph.add_edge(START,"node")
graph.add_edge("node",END)

app=graph.compile()

for i,chunks in enumerate(app.stream({"messages":"Count From 1 to 20"},stream_mode="messages")):
    print(f"Message {i}: {chunks}")

Message 0: (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--01a1da0e-3f70-7280-add6-764024dd501f', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'node', 'langgraph_triggers': ('branch:to:node',), 'langgraph_path': ('__pregel_pull', 'node'), 'langgraph_checkpoint_ns': 'node:72020316-d996-04ea-fa89-ee839707c93f', 'checkpoint_ns': 'node:72020316-d996-04ea-fa89-ee839707c93f', 'ls_provider': 'openai', 'ls_model_name': 'gpt-4o-mini', 'ls_model_type': 'chat', 'ls_temperature': 0.0, 'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7', 'langchain-openai': '1.7.2'}})
Message 1: (AIMessageChunk(content='Sure', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--01a1da0e-3f70-7280-add6-764024dd501f', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'ls_integration': 'langchain_chat_model', 